# Video2X Colab notebook (fork)

Builds Video2X from the `adeshboudhnicedigitals/video2x` fork, which adds a pipelined decode/process/encode loop (`--queue-size`), and upscales anime video on a Colab GPU.

**Goal:** 1080p output with the best picture quality. 4K and 8K are out of scope for now.

Settings are plain Python variables near the top of each cell: edit the values, then run the cell.

| Section | Cells |
|---|---|
| 1. Setup | 1.1 GPU check, 1.2 build, 1.3 input file, 1.4 resource monitor (optional) |
| 2. Upscale | 2.1 upscale a video |
| 3. Measure | 3.1 speed test, 3.2 model comparison, 3.3 resource monitor results |
| 4. Experimental | 4.1 frame interpolation (broken on Colab), 4.2 to 4.4 PyTorch/CUDA backend tests |

Based on the [Video2X](https://github.com/k4yt3x/video2x) Colab notebook for version 6.2.0. Licensed under the ISC License: https://www.isc.org/licenses/. \
Copyright 2022-2024 K4YT3X and contributors.

## 1. Setup

Run 1.1 to 1.3 once per runtime session, in order.

In [ ]:
# 1.1 GPU check
#
# You'll need to select a GPU runtime for Video2X to work. On the menu bar at the top, select:

# Runtime > Change runtime type > Hardware accelerator

# Change "Hardware accelerator" to GPU and click "Save." Then run this cell to see the detailed
# information about the GPU allocated to you (optional).

!nvidia-smi

In [ ]:
# 1.2 Build Video2X from source
#
# This step builds Video2X from the repository and branch below instead of installing the released
# package.

# - The first build takes roughly 10-20 minutes on Colab. You only need to run it once per runtime
#   session.
# - Fork submodules (ncnn, spdlog and the model wrappers) are fetched automatically; the large
#   Boost submodule is not needed because the system Boost is used.

repo_url = "https://github.com/adeshboudhnicedigitals/video2x.git"
branch = "master"

!add-apt-repository -y ppa:ubuntuhandbook1/ffmpeg7
!apt-get update
!DEBIAN_FRONTEND=noninteractive apt-get install -y --no-install-recommends libvulkan1 nvidia-driver-535 ffmpeg git build-essential cmake ninja-build libavcodec-dev libavdevice-dev libavfilter-dev libavformat-dev libavutil-dev libswscale-dev libvulkan-dev glslang-tools libomp-dev libboost-program-options-dev

!rm -rf /content/video2x-src
!git clone --branch {branch} {repo_url} /content/video2x-src
!cd /content/video2x-src && git submodule update --init --recursive -- third_party/ncnn third_party/spdlog third_party/librealesrgan_ncnn_vulkan third_party/librealcugan_ncnn_vulkan third_party/librife_ncnn_vulkan

!cd /content/video2x-src && cmake -G Ninja -B build -S . -DVIDEO2X_USE_EXTERNAL_NCNN=OFF -DVIDEO2X_USE_EXTERNAL_SPDLOG=OFF -DCMAKE_CXX_COMPILER=g++ -DCMAKE_BUILD_TYPE=Release -DCMAKE_INSTALL_PREFIX=/usr
!cd /content/video2x-src && cmake --build build --config Release --target install --parallel
!ldconfig

# Confirm that the freshly built binary has the pipelining option
!video2x --help | grep -A1 "queue-size"

# Report whether this FFmpeg can encode on the GPU (NVENC); cell 2.1 can use it with codec h264_nvenc or hevc_nvenc
!ffmpeg -hide_banner -encoders | grep -i nvenc || echo "No NVENC encoders found in this FFmpeg build"

%env VK_ICD_FILENAMES=/usr/share/vulkan/icd.d/nvidia_icd.json

In [ ]:
# 1.3 Input file
#
# Picks the video to process and the output directory.
# - Files in /content live on this runtime's local disk and are deleted when the runtime
#   disconnects or is reset. Download your output before then.
# - Put a video in /content (for example with !wget), or set upload_new_file = True to use
#   Colab's upload dialog.

import os
import pathlib

input_path = ""                 # video to process; "" uses the only video found under /content
output_dir = "/content/output"  # where processed videos are written
upload_new_file = False         # True opens the upload dialog first; set it back to False afterwards

media_dir = pathlib.Path("/content")
output_dir = pathlib.Path(output_dir)
output_dir.mkdir(parents=True, exist_ok=True)

VIDEO_SUFFIXES = {".mp4", ".mkv", ".mov", ".avi", ".webm", ".m4v", ".flv", ".wmv", ".ts"}
SKIP_DIRS = {"sample_data", "video2x-src", "pipeline_bench", "speed_test", "model_comparison",
             "shrunk", "benchmark_output", "__pycache__"}

def find_videos():
    found = []
    for root, dirs, names in os.walk(media_dir):
        dirs[:] = [
            d for d in dirs
            if not d.startswith(".")
            and d not in SKIP_DIRS
            and pathlib.Path(root, d) != output_dir
        ]
        for name in names:
            if pathlib.Path(name).suffix.lower() in VIDEO_SUFFIXES:
                found.append(pathlib.Path(root, name))
    return sorted(found)

if upload_new_file:
    from google.colab import files
    uploaded = files.upload()
    if uploaded:
        input_path = str(media_dir / next(iter(uploaded)))
    else:
        print("Upload canceled.")

videos = find_videos()
if videos:
    print("Videos found:")
    for video in videos:
        print(f"  {video}")

if input_path.strip():
    input_file = pathlib.Path(input_path.strip())
    if not input_file.is_file():
        raise FileNotFoundError(f"input_path does not point to a file: {input_file}")
elif len(videos) == 1:
    input_file = videos[0]
elif not videos:
    raise FileNotFoundError("No videos found under /content. Put one there, or set upload_new_file = True.")
else:
    raise ValueError("Several videos found. Set input_path to one of the paths listed above.")

print(f"Selected input file: {input_file}")

In [ ]:
# 1.4 Resource monitor: start (optional)
#
# Starts a background logger that records GPU and RAM usage while Video2X runs. Run this cell
# before a job, then run cell 3.3 afterwards to see the results.

# - It samples every `interval_seconds` and writes a CSV to `log_file`: GPU utilization, GPU
#   memory, SM clock, temperature, power draw, throttle reasons, system RAM, the `video2x`
#   process's memory and the CPU load.
# - Re-running this cell restarts the log. The logger stops when you run cell 3.3 with
#   `stop_monitor = True`, or when the runtime ends.

import csv
import os
import subprocess
import threading
import time

interval_seconds = 5
log_file = "/content/monitor.csv"

# Stop a previous monitor if this cell is re-run
if globals().get("monitor_stop") is not None:
    monitor_stop.set()
    if globals().get("monitor_thread") is not None:
        monitor_thread.join(timeout=10)

GPU_FIELDS = [
    "utilization.gpu", "utilization.memory", "memory.used", "clocks.sm",
    "clocks.max.sm", "temperature.gpu", "power.draw", "power.limit",
]

def query_gpu(fields):
    return subprocess.run(
        ["nvidia-smi", f"--query-gpu={','.join(fields)}", "--format=csv,noheader,nounits"],
        capture_output=True, text=True,
    )

# The throttle-reason field was renamed in newer drivers; use whichever works
throttle_field = None
for candidate in ("clocks_throttle_reasons.active", "clocks_event_reasons.active"):
    if query_gpu(GPU_FIELDS + [candidate]).returncode == 0:
        throttle_field = candidate
        break

def num(text):
    try:
        return float(text)
    except (TypeError, ValueError):
        return None

def read_ram_mb():
    values = {}
    with open("/proc/meminfo") as f:
        for line in f:
            key, rest = line.split(":", 1)
            values[key] = int(rest.split()[0]) / 1024
    return round(values["MemTotal"] - values["MemAvailable"], 1), round(values["MemTotal"], 1)

def video2x_rss_mb():
    total = 0.0
    for pid in os.listdir("/proc"):
        if not pid.isdigit():
            continue
        try:
            with open(f"/proc/{pid}/comm") as f:
                if f.read().strip() != "video2x":
                    continue
            with open(f"/proc/{pid}/status") as f:
                for line in f:
                    if line.startswith("VmRSS:"):
                        total += int(line.split()[1]) / 1024
        except OSError:
            pass
    return round(total, 1)

HEADER = [
    "elapsed_s", "gpu_util_pct", "gpu_mem_util_pct", "gpu_mem_used_mb", "sm_clock_mhz",
    "sm_clock_max_mhz", "temp_c", "power_w", "power_limit_w", "throttle_reasons",
    "ram_used_mb", "ram_total_mb", "video2x_rss_mb", "load1",
]

def monitor_loop(stop, path, interval):
    started = time.time()
    fields = GPU_FIELDS + ([throttle_field] if throttle_field else [])
    with open(path, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(HEADER)
        f.flush()
        while not stop.is_set():
            gpu = [None] * 9
            result = query_gpu(fields)
            if result.returncode == 0 and result.stdout.strip():
                parts = [p.strip() for p in result.stdout.strip().splitlines()[0].split(",")]
                gpu = [num(p) for p in parts[:8]]
                gpu.append(parts[8] if len(parts) > 8 else None)
            ram_used, ram_total = read_ram_mb()
            writer.writerow(
                [round(time.time() - started, 1)] + gpu
                + [ram_used, ram_total, video2x_rss_mb(), os.getloadavg()[0]]
            )
            f.flush()
            stop.wait(interval)

monitor_stop = threading.Event()
monitor_thread = threading.Thread(
    target=monitor_loop, args=(monitor_stop, log_file, interval_seconds), daemon=True
)
monitor_thread.start()
print(f"Monitoring every {interval_seconds} s -> {log_file}")
if throttle_field is None:
    print("Note: this driver does not report throttle reasons; that column will be empty.")

## 2. Upscale

In [ ]:
# 2.1 Upscale a video
#
# Set the values below and run this cell.

# - Remember free sessions can run for at most 12 hours
#   (https://research.google.com/colaboratory/faq.html#idle-timeouts). If you exceed this limit
#   your progress could be lost. You can get Colab Pro/Pro+
#   (https://colab.research.google.com/signup/pricing) for longer runtimes.
# - It may be a good idea to upscale a small sample of your video to see how well you like the
#   results before processing the whole video.
# - The model enlarges the input height (or the shrink-first height) by the scale factor: a 1080p
#   video at scale 2 becomes 4K. `output_height` then resizes each upscaled frame to the final
#   size before it is encoded. The default is scale 2 resized to 1080p, which is the current goal.

import pathlib
import subprocess
import time

if 'input_file' not in globals() or input_file is None:
    raise NameError("Input file not specified. Did you forget to run cell 1.3?")

if 'output_dir' not in globals():
    raise NameError("Output directory not specified. Did you forget to run cell 1.3?")

# --- Filter Select ---
# - `libplacebo` currently has compatibility issues with Colab's environment.
# - `realesrgan` and `realcugan` are neural upscalers for anime/cartoon video. In our tests they
#   ran at about the same speed on this setup, so compare their look with cell 3.2.
# - Choose `realesrgan` with the `realesrgan-plus` model for real-life videos.

filter = "realesrgan"  # one of: realesrgan, realcugan, libplacebo

# --- Shrink First (optional) ---
# - If your file is labelled 1080p but its real detail is lower (for example an older show that
#   was upscaled before release), shrinking it to 720p first can look about the same while taking
#   less than half the time. Set to `0` to turn this off.
# - A temporary copy is made with a high-quality resize (audio and subtitles are kept), upscaled,
#   and then deleted. A video that is already at or below this height is left alone.

shrink_first_height = 0

# --- Upscale Factor ---
# - Used by `realesrgan` and `realcugan`. `realesr-animevideov3` supports 2x, 3x and 4x; other
#   Real-ESRGAN models support only 4x. Real-CUGAN support depends on the model (see below).

scale = 2  # 2 to 4

# --- Output Height ---
# - Used by `realesrgan` and `realcugan`. Each upscaled frame is resized to this height with a
#   high-quality (Lanczos) resize before encoding; the width keeps the aspect ratio. Set to `0`
#   to keep the full upscaled size.
# - This does not make the upscaling faster (the model still runs at the full scale), but the
#   encoder has less work and the file is a normal 1080p video.

output_height = 1080

# --- RealESRGAN Options ---
# - Choose `realesr-animevideov3` or `realesrgan-plus-anime` for anime/cartoon videos.
# - Choose `realesrgan-plus` for real-life videos.

model = "realesr-animevideov3"  # one of: realesrgan-plus-anime, realesrgan-plus, realesr-animevideov3

# --- Real-CUGAN Options ---
# - `models-se` supports 2x, 3x and 4x; `models-pro` supports 2x and 3x; `models-nose` supports 2x
#   only.
# - `cugan_noise`: `conservative` keeps the original look, `none` applies no denoising, and `1`,
#   `2`, `3` denoise progressively harder (which can smooth fine texture). Not every level exists
#   for every model: `models-se` at 2x has all of them; at 3x and 4x, and for `models-pro`, only
#   `conservative`, `none` and `3`; `models-nose` has `none` only.

cugan_model = "models-se"  # one of: models-se, models-pro, models-nose
cugan_noise = "conservative"  # one of: conservative, none, 1, 2, 3

# --- libplacebo Options ---

# - `anime4k-v4-a`: Optimized for 1080p anime
# - `anime4k-v4-b`: Optimized for 720p anime
# - `anime4k-v4-c`: Optimized for 480p anime
# - `anime4k-v4-a+a`: Higher quality version of mode A
# - `anime4k-v4-b+b`: Higher quality version of mode B
# - `anime4k-v4-c+a`: Higher quality version of mode C
# - `anime4k-v4.1-gan`: Experimental SRGAN shaders; delivers the highest quality but is slower
#   than standard modes
# - `width` and `height` are the output size, and only apply to `libplacebo`.

shader = "anime4k-v4-a"  # one of: anime4k-v4-a, anime4k-v4-a+a, anime4k-v4-b, anime4k-v4-b+b, anime4k-v4-c, anime4k-v4-c+a, anime4k-v4.1-gan
width = 1920
height = 1080

# --- Advanced Options ---

# You typically don't need to touch these if you don't know what they do.

# - `preset` and `crf`: slower x264 presets and a lower `crf` give a better picture for the file
#   size but cost CPU time, and Colab has only 2 cores. In our test the encoder took 3% of the
#   time at 4K with `veryfast`; at 1080p output it has a quarter of that work, so `slow` with
#   `crf` 18 is the default. If cell 3.1 shows the encoder holding the GPU back, use `medium`
#   or `veryfast`.
# - To encode on the GPU, set `codec` to `h264_nvenc` or `hevc_nvenc` (cell 1.2 prints whether
#   this FFmpeg has them). Then `preset` must be `p1` to `p7` (`p7` is the slowest and best), and
#   `crf` is used as the NVENC `cq` quality value (lower is better). NVENC is usually less
#   efficient than x264 at the same size, so compare the look.

codec = "libx264"
preset = "slow"
pixfmt = "auto"
bitrate = 0
crf = 18
loglevel = "info"  # one of: trace, debug, info, warn, error, critical, none

# - `queue_size`: frames buffered between the decode, GPU and encode stages (default 4). Set it to
#   `0` for the original, non-pipelined behavior. Each buffered 4K frame uses about 12-25 MB of
#   RAM.
queue_size = 4

# - `tile_size`: `realesrgan` only. The frame is processed in square tiles of this many pixels,
#   each with a 10-pixel border that is computed and thrown away. `0` picks the size from the GPU
#   memory (200 on a T4, so a 1080p frame is 60 tiles with about 21% extra work). Larger tiles
#   waste less but use more GPU memory; cell 3.1 measures a few sizes. The picture changes very
#   slightly near tile edges.
# - Measured on a T4 at 1080p x2: 400 was 8% faster than the automatic 200; 1000 and 1920 were
#   only 1% faster. 400 is the default here.
tile_size = 400

# - `benchmark`: run the processing only, with no encoding, to measure the speed ceiling. A
#   throwaway output file is made and deleted afterwards. Use it to tell whether the encoder is
#   the limit (compare the fps with a normal run).
benchmark = False

# --- Check that the chosen model supports the chosen scale and noise level ---
CUGAN_NOISE = {"conservative": -1, "none": 0, "1": 1, "2": 2, "3": 3}
CUGAN_SUPPORT = {
    "models-se": {2: {-1, 0, 1, 2, 3}, 3: {-1, 0, 3}, 4: {-1, 0, 3}},
    "models-pro": {2: {-1, 0, 3}, 3: {-1, 0, 3}},
    "models-nose": {2: {0}},
}

if filter == "realesrgan":
    supported_scales = {2, 3, 4} if model == "realesr-animevideov3" else {4}
    if scale not in supported_scales:
        raise ValueError(f"{model} supports only {sorted(supported_scales)}x, not {scale}x. Change the scale factor.")
elif filter == "realcugan":
    noise_value = CUGAN_NOISE[cugan_noise]
    supported = CUGAN_SUPPORT[cugan_model]
    if scale not in supported:
        raise ValueError(f"{cugan_model} supports only {sorted(supported)}x, not {scale}x. Change the scale factor or the model.")
    if noise_value not in supported[scale]:
        names = [name for name, value in CUGAN_NOISE.items() if value in supported[scale]]
        raise ValueError(f"{cugan_model} at {scale}x does not have the '{cugan_noise}' noise level. Choose one of: {', '.join(names)}.")

output_file = output_dir / f"{input_file.stem}.processed{input_file.suffix}"
if benchmark:
    output_file = pathlib.Path("/content/benchmark_output") / f"{input_file.stem}.benchmark{input_file.suffix}"
    output_file.parent.mkdir(exist_ok=True)
    output_file.unlink(missing_ok=True)
if output_file.exists():
    raise FileExistsError(f"Cannot continue: file {output_file} already exists")
print(f"Output will be written to: {output_file}")

def probe_size(video_path):
    values = subprocess.run(
        ["ffprobe", "-v", "error", "-select_streams", "v:0", "-show_entries", "stream=width,height",
         "-of", "csv=p=0", str(video_path)],
        capture_output=True, text=True, check=True,
    ).stdout.strip().split(",")
    return int(values[0]), int(values[1])

# --- Optionally shrink the input first ---
video2x_input = input_file
shrunk_file = None
input_width, input_height = probe_size(input_file)
if shrink_first_height > 0:
    if input_height <= shrink_first_height:
        print(f"The input is {input_height}p, which is not above {shrink_first_height}p, so it is used as it is.")
    else:
        shrunk_dir = pathlib.Path("/content/shrunk")
        shrunk_dir.mkdir(exist_ok=True)
        shrunk_file = shrunk_dir / f"{input_file.stem}.shrunk{input_file.suffix}"
        shrunk_file.unlink(missing_ok=True)

        def build_shrink_command(with_subtitles):
            shrink_command = ["ffmpeg", "-y", "-loglevel", "error", "-i", str(input_file),
                              "-map", "0:v:0", "-map", "0:a?"]
            if with_subtitles:
                shrink_command += ["-map", "0:s?"]
            shrink_command += ["-vf", f"scale=-2:{shrink_first_height}:flags=lanczos",
                               "-c:v", "libx264", "-preset", "veryfast", "-crf", "10",
                               "-pix_fmt", "yuv420p", "-c:a", "copy"]
            if with_subtitles:
                shrink_command += ["-c:s", "copy"]
            return shrink_command + [str(shrunk_file)]

        print(f"Shrinking {input_height}p to {shrink_first_height}p first...")
        started = time.time()
        result = subprocess.run(build_shrink_command(True), capture_output=True, text=True)
        if result.returncode != 0:
            # Some subtitle formats cannot be copied into this container; retry without them
            print("Copying the subtitles failed; retrying without subtitles.")
            result = subprocess.run(build_shrink_command(False), capture_output=True, text=True)
        if result.returncode != 0:
            print(result.stderr[-1500:])
            raise RuntimeError("Shrinking the input failed")
        print(f"  done in {time.time() - started:.0f} s")
        video2x_input = shrunk_file
        input_width, input_height = probe_size(shrunk_file)

if filter in ("realesrgan", "realcugan"):
    upscaled_width, upscaled_height = input_width * scale, input_height * scale
    if output_height > 0:
        final_width = round(input_width * output_height / input_height / 2) * 2
        print(f"Input to the upscaler: {input_width}x{input_height}; upscaled to {upscaled_width}x{upscaled_height}, "
              f"then resized to {final_width}x{output_height}")
    else:
        print(f"Input to the upscaler: {input_width}x{input_height}; expected output: {upscaled_width}x{upscaled_height}")

command = [
    "video2x",

    # Common options
    "--input",
    f"\"{video2x_input}\"",
    "--output",
    f"\"{output_file}\"",
    "--processor",
    filter,

    # Encoder options
    "--codec",
    codec,
    "--bit-rate",
    str(bitrate),
    "--log-level",
    loglevel,
    "--queue-size",
    str(queue_size),
]

if codec.endswith("_nvenc"):
    # NVENC has no crf; constant quality is rc=vbr with cq and a zero target bitrate
    command.extend(["-e", f"preset={preset}", "-e", "rc=vbr", "-e", f"cq={crf}"])
else:
    command.extend(["-e", f"preset={preset}", "-e", f"crf={crf}"])

if benchmark:
    command.append("--benchmark")

# RealESRGAN options
if filter == "realesrgan":
    command.extend([
        "--realesrgan-model",
        model,
        "--scaling-factor",
        str(scale)
    ])
    if tile_size > 0:
        command.extend(["--realesrgan-tile-size", str(tile_size)])

# Real-CUGAN options
elif filter == "realcugan":
    command.extend([
        "--realcugan-model",
        cugan_model,
        "--scaling-factor",
        str(scale),
        f"--noise-level={CUGAN_NOISE[cugan_noise]}"
    ])

# Final size for the neural upscalers
if filter in ("realesrgan", "realcugan") and output_height > 0:
    command.extend(["--height", str(output_height)])

# libplacebo options
if filter == "libplacebo":
    command.extend([
        "--width",
        str(width),
        "--height",
        str(height),
        "--libplacebo-shader",
        f"\"{shader}\""
    ])

if pixfmt != "auto":
    command.extend([
        "--pix-fmt",
        pixfmt
    ])

try:
    run_started = time.time()
    !{' '.join(command)}
    print(f"Wall time: {time.time() - run_started:.0f} s")
finally:
    if benchmark:
        output_file.unlink(missing_ok=True)
    if shrunk_file is not None:
        shrunk_file.unlink(missing_ok=True)
        print("Removed the temporary shrunk copy.")

## 3. Measure

Short tests on a sample of the input. Start with 3.1 to see what limits the speed.

In [ ]:
# 3.1 Speed test: where does the time go?
#
# Cuts a short clip from the selected input and upscales it several ways with the same model:
#   A  no encoding (--benchmark)       the ceiling: decode, colour conversion and GPU only
#   B  x264, pipelined                 the normal job
#   C  x264, serial (--queue-size 0)   the original loop, to show what pipelining gains
#   D  NVENC, pipelined                encoding on the GPU instead of the CPU
#   T  tile sizes, no encoding         the same as A with larger Real-ESRGAN tiles
#
# Results so far (T4, 1080p x2 to 4K, x264 veryfast): A 1.35 fps, B 1.31, C 1.26, and D failed
# because this build's FFmpeg libraries have no NVENC. The encoder and the pipelining are
# therefore settled, so C and D are off by default. At 1080p output with x264 slow: A 1.37 fps,
# B 1.23 (90% of A), tile 400 1.48, tile 1000 1.38, tile 1920 1.39. The tile sizes listed below
# narrow down the best size around 400.
#
# How to read it:
# - A close to B: encoding is not what limits the job, so NVENC will not help.
# - A well above B: the CPU encoder holds the GPU back; use a faster x264 preset.
# - A T run above A: put that tile size in cell 2.1. A failed T run usually means the GPU ran
#   out of memory at that size.
# - B and C use the same encoder and settings, so their frames should be identical. If the
#   hashes differ, that may come from the encoder rather than the pipeline.
# - The GPU slows down as it heats up, so one untimed warm-up run is done first and the GPU
#   clock, temperature and power are printed after every run. Compare runs with similar clocks.
# - The clip is cut without re-encoding, so it starts at the nearest keyframe before clip_start.
# - Run cell 1.3 first.

import pathlib
import re
import subprocess
import time

if 'input_file' not in globals() or input_file is None:
    raise NameError("Input file not specified. Did you forget to run cell 1.3?")

clip_start = 60                     # seconds into the input
clip_seconds = 5                    # 5 s is about 150 frames, roughly 2 minutes per run on a T4
test_model = "realesr-animevideov3"
test_scale = 2
test_output_height = 1080           # resize the upscaled frames to this height; 0 keeps the full size
x264_preset = "slow"
x264_crf = 18
run_serial = False                  # run C
nvenc_codec = ""                    # "h264_nvenc", "hevc_nvenc", or "" to skip run D
tile_sizes = [300, 400, 500, 700]   # Real-ESRGAN tile sizes for the T runs; [] skips them
nvenc_preset = "p7"                 # p1 (fastest) to p7 (best quality)
nvenc_cq = 19                       # NVENC quality value; lower is better
pipelined_queue_size = 4
warm_up = True                      # one untimed run first, so timed runs start with a warm GPU

work = pathlib.Path("/content/speed_test")
work.mkdir(exist_ok=True)
clip = work / f"clip{input_file.suffix}"
clip.unlink(missing_ok=True)

duration = float(subprocess.run(
    ["ffprobe", "-v", "error", "-show_entries", "format=duration", "-of", "default=nw=1:nk=1", str(input_file)],
    capture_output=True, text=True, check=True,
).stdout.strip())
clip_begin = clip_start
if clip_start + clip_seconds > duration:
    clip_begin = max(0.0, duration - clip_seconds)
    print(f"The input is {duration:.1f} s long, so the clip starts at {clip_begin:.1f} s instead of {clip_start} s.")

subprocess.run(
    ["ffmpeg", "-y", "-loglevel", "error", "-ss", str(clip_begin), "-t", str(clip_seconds),
     "-i", str(input_file), "-c", "copy", str(clip)],
    check=True,
)

def gpu_state():
    try:
        result = subprocess.run(
            ["nvidia-smi", "--query-gpu=clocks.sm,temperature.gpu,power.draw", "--format=csv,noheader,nounits"],
            capture_output=True, text=True,
        )
    except OSError:
        return "unavailable"
    return result.stdout.strip().replace("\n", " ") if result.returncode == 0 else "unavailable"

x264_options = ["--codec", "libx264", "-e", f"preset={x264_preset}", "-e", f"crf={x264_crf}"]
# NVENC has no crf; constant quality is rc=vbr with cq
nvenc_options = ["--codec", nvenc_codec, "-e", f"preset={nvenc_preset}", "-e", "rc=vbr", "-e", f"cq={nvenc_cq}"]

def run_test(name, encoder_options, test_queue_size, benchmark=False, extra=()):
    out = work / f"out_{name}.mkv"
    out.unlink(missing_ok=True)
    command = [
        "video2x", "--input", str(clip), "--output", str(out),
        "--processor", "realesrgan", "--realesrgan-model", test_model,
        "--scaling-factor", str(test_scale),
        "--log-level", "info", "--no-progress",
        "--queue-size", str(test_queue_size),
    ] + encoder_options + list(extra)
    if test_output_height > 0:
        command += ["--height", str(test_output_height)]
    if benchmark:
        command.append("--benchmark")
    started = time.time()
    result = subprocess.run(command, capture_output=True, text=True)
    seconds = time.time() - started
    text = result.stdout + result.stderr
    if benchmark:
        out.unlink(missing_ok=True)
    speed = re.search(r"Average processing speed:\s*([0-9.]+)\s*FPS", text)
    count = re.search(r"Total frames processed:\s*(\d+)", text)
    if result.returncode != 0 or speed is None or float(speed.group(1)) <= 0:
        print(f"  failed. Output tail:\n{text[-800:]}")
        return None
    return {
        "out": None if benchmark else out,
        "seconds": seconds,
        "fps": float(speed.group(1)),
        "frames": int(count.group(1)) if count else None,
        "gpu": gpu_state(),
    }

def frame_lines(path):
    result = subprocess.run(
        ["ffmpeg", "-loglevel", "error", "-i", str(path), "-map", "0:v", "-f", "framemd5", "-"],
        capture_output=True, text=True, check=True,
    )
    return [line.strip() for line in result.stdout.splitlines() if line and not line.startswith("#")]

tests = [
    ("A", "no encoding (--benchmark)", x264_options, pipelined_queue_size, True, []),
    ("B", f"x264 {x264_preset}, pipelined", x264_options, pipelined_queue_size, False, []),
]
if run_serial:
    tests.append(("C", f"x264 {x264_preset}, serial", x264_options, 0, False, []))
if nvenc_codec:
    tests.append(("D", f"{nvenc_codec}, pipelined", nvenc_options, pipelined_queue_size, False, []))
for size in tile_sizes:
    tests.append((f"T{size}", f"tile size {size}, no encoding", x264_options, pipelined_queue_size, True,
                  ["--realesrgan-tile-size", str(size)]))

if warm_up:
    print("Warming up the GPU (untimed)...")
    run_test("warmup", x264_options, pipelined_queue_size, benchmark=True)

speed_results = {}
for key, label, encoder_options, test_queue_size, benchmark, extra in tests:
    print(f"Running {key}: {label}...")
    speed_results[key] = run_test(key, encoder_options, test_queue_size, benchmark, extra)
    if speed_results[key]:
        print(f"  {speed_results[key]['fps']:.2f} fps in {speed_results[key]['seconds']:.0f} s")

print()
print(f"{'run':<40}{'fps':>7}{'seconds':>9}{'frames':>8}   GPU clock MHz, temp C, power W")
for key, label, _, _, _, _ in tests:
    row = speed_results[key]
    name = f"{key}  {label}"
    if row is None:
        print(f"{name:<40} failed")
        continue
    frames = "?" if row["frames"] is None else row["frames"]
    print(f"{name:<40}{row['fps']:>7.2f}{row['seconds']:>9.0f}{frames:>8}   {row['gpu']}")

a, b, c, d = (speed_results.get(key) for key in "ABCD")
print()
if a and b:
    print(f"Encoding: the normal job (B) reaches {b['fps'] / a['fps'] * 100:.0f}% of the no-encode ceiling (A).")
if b and c:
    identical = frame_lines(b["out"]) == frame_lines(c["out"])
    print(f"Pipelining: B is {b['fps'] / c['fps']:.2f}x the serial loop (C); "
          f"frame count, timestamps and pixel hashes are {'IDENTICAL' if identical else 'DIFFERENT'}.")
if b and d:
    size_b = b["out"].stat().st_size / 1024 / 1024
    size_d = d["out"].stat().st_size / 1024 / 1024
    print(f"NVENC: D is {d['fps'] / b['fps']:.2f}x the x264 job (B). File sizes: x264 {size_b:.1f} MB, "
          f"{nvenc_codec} {size_d:.1f} MB. Compare the picture before choosing NVENC.")
for size in tile_sizes:
    t = speed_results.get(f"T{size}")
    if a and t:
        print(f"Tile size {size}: {t['fps'] / a['fps']:.2f}x the default tile size (A).")
print(f"Outputs are kept in {work} for inspection.")

In [ ]:
# 3.2 Compare models for picture quality
#
# Upscales a few seconds of your video with several models and settings, then shows the same frame
# from each side by side with the time each one took. Pick the look you like here, then use that
# model in cell 2.1 for the whole video. Run cell 1.3 first.

# - The top row shows the whole frame; the bottom row zooms into the same region of each result so
#   differences in lines, noise and blur are visible. Panels are resized to the same display size,
#   so a result with fewer pixels will look softer, as it would on screen.
# - Source (Lanczos) is a plain resize of the original frame, with no AI. A model has to look
#   better than this to be worth the time.
# - All models run at x2 here. Real-CUGAN's `conservative` keeps the original look; higher
#   `denoise` levels remove more noise and compression artifacts but can smooth fine texture.
# - `try_720p_first` shrinks the sample to 720p before upscaling. If the real detail of your file
#   is only 720p, it can look as good for much less GPU time.
# - Times include a few seconds of start-up per run, so they are slightly pessimistic on short
#   samples. Real-CUGAN speeds have not been measured on this setup before; expect it to be slower
#   than Real-ESRGAN.
# - The comparison image is also saved to `/content/output/model_comparison.png`.

import json
import pathlib
import subprocess
import time

import matplotlib.pyplot as plt
from PIL import Image

if 'input_file' not in globals() or input_file is None:
    raise NameError("Input file not specified. Did you forget to run cell 1.3?")

sample_start = 60
sample_seconds = 2
still_time = 1.0
crop_center_x = 0.5  # 0.1 to 0.9
crop_center_y = 0.4  # 0.1 to 0.9
crop_width_fraction = 0.2  # 0.1 to 0.5

try_realesrgan_animevideov3 = True
try_realcugan_conservative = True
try_realcugan_denoise1 = True
try_realcugan_denoise3 = False
try_720p_first = True

work = pathlib.Path("/content/model_comparison")
work.mkdir(exist_ok=True)
save_dir = pathlib.Path(globals().get("output_dir", "/content/output"))
save_dir.mkdir(parents=True, exist_ok=True)

def run(command, **kwargs):
    return subprocess.run(command, capture_output=True, text=True, **kwargs)

# --- Cut the sample (clamped to the input's duration) ---
duration = float(run(
    ["ffprobe", "-v", "error", "-show_entries", "format=duration", "-of", "default=nw=1:nk=1", str(input_file)],
    check=True,
).stdout.strip())
begin = sample_start
if sample_start + sample_seconds > duration:
    begin = max(0.0, duration - sample_seconds)
    print(f"The input is {duration:.1f} s long, so the sample starts at {begin:.1f} s instead of {sample_start} s.")

sample = work / "sample.mkv"
sample.unlink(missing_ok=True)
run(["ffmpeg", "-y", "-loglevel", "error", "-ss", str(begin), "-t", str(sample_seconds),
     "-i", str(input_file), "-c", "copy", "-an", str(sample)], check=True)

def probe(path):
    info = json.loads(run(
        ["ffprobe", "-v", "error", "-select_streams", "v:0", "-count_frames",
         "-show_entries", "stream=width,height,nb_read_frames", "-of", "json", str(path)],
        check=True,
    ).stdout)["streams"][0]
    return int(info["width"]), int(info["height"]), int(info["nb_read_frames"])

src_width, src_height, src_frames = probe(sample)
if src_frames == 0:
    raise RuntimeError("The sample has no frames; check sample_start and sample_seconds")
print(f"Sample: {src_width}x{src_height}, {src_frames} frames")

still_at = min(still_time, max(0.0, sample_seconds - 0.2))

# --- Candidates: (label, input clip, extra video2x arguments) ---
sample_720 = None
if try_720p_first:
    sample_720 = work / "sample_720p.mkv"
    sample_720.unlink(missing_ok=True)
    run(["ffmpeg", "-y", "-loglevel", "error", "-i", str(sample), "-vf", "scale=-2:720:flags=lanczos",
         "-c:v", "libx264", "-crf", "10", "-preset", "veryfast", "-an", str(sample_720)], check=True)

realesrgan = ["--processor", "realesrgan", "--realesrgan-model", "realesr-animevideov3", "--scaling-factor", "2"]
def cugan(noise):
    return ["--processor", "realcugan", "--realcugan-model", "models-se", "--scaling-factor", "2",
            f"--noise-level={noise}"]

candidates = []
if try_realesrgan_animevideov3:
    candidates.append(("Real-ESRGAN animevideov3", sample, realesrgan))
if try_realcugan_conservative:
    candidates.append(("Real-CUGAN conservative", sample, cugan(-1)))
if try_realcugan_denoise1:
    candidates.append(("Real-CUGAN denoise 1", sample, cugan(1)))
if try_realcugan_denoise3:
    candidates.append(("Real-CUGAN denoise 3", sample, cugan(3)))
if try_720p_first:
    candidates.append(("animevideov3 from 720p", sample_720, realesrgan))
if not candidates:
    raise ValueError("Select at least one model to compare.")

# --- Run each candidate and grab the same frame ---
results = []
for index, (label, clip, arguments) in enumerate(candidates):
    out = work / f"result_{index}.mkv"
    out.unlink(missing_ok=True)
    print(f"Running {label}...")
    started = time.time()
    result = run(
        ["video2x", "--input", str(clip), "--output", str(out), "--codec", "libx264",
         "-e", "preset=veryfast", "-e", "crf=14", "--log-level", "warn", "--no-progress",
         "--queue-size", "4"] + arguments
    )
    seconds = time.time() - started
    if result.returncode != 0 or not out.exists():
        print(f"  failed, skipping. Output tail:\n{(result.stdout + result.stderr)[-800:]}")
        continue
    still = work / f"still_{index}.png"
    still.unlink(missing_ok=True)
    run(["ffmpeg", "-y", "-loglevel", "error", "-ss", str(still_at), "-i", str(out),
         "-frames:v", "1", str(still)], check=True)
    results.append((f"{label}\n{seconds:.0f} s ({src_frames / seconds:.2f} fps)", Image.open(still).convert("RGB")))
    print(f"  done in {seconds:.1f} s")

if not results:
    raise RuntimeError("No model produced a result.")

# --- Plain Lanczos resize of the source frame as the no-AI reference ---
ref_width, ref_height = results[0][1].size
reference_path = work / "still_reference.png"
reference_path.unlink(missing_ok=True)
run(["ffmpeg", "-y", "-loglevel", "error", "-ss", str(still_at), "-i", str(sample), "-frames:v", "1",
     "-vf", f"scale={ref_width}:{ref_height}:flags=lanczos", str(reference_path)], check=True)
panels = [("Source (Lanczos)\nno AI", Image.open(reference_path).convert("RGB"))] + results

# --- Show whole frames (top) and zoomed crops (bottom) ---
display_width = 640
def crop_region(image):
    width, height = image.size
    crop_w = int(width * crop_width_fraction)
    crop_h = int(crop_w * height / width)
    left = min(max(int(width * crop_center_x - crop_w / 2), 0), width - crop_w)
    top = min(max(int(height * crop_center_y - crop_h / 2), 0), height - crop_h)
    return image.crop((left, top, left + crop_w, top + crop_h))

figure, axes = plt.subplots(2, len(panels), figsize=(4.2 * len(panels), 5.6), squeeze=False)
for column, (title, image) in enumerate(panels):
    full = image.resize((display_width, int(display_width * image.height / image.width)), Image.LANCZOS)
    zoom = crop_region(image)
    zoom = zoom.resize((display_width, int(display_width * zoom.height / zoom.width)), Image.BICUBIC)
    axes[0][column].imshow(full)
    axes[0][column].set_title(title, fontsize=9)
    axes[1][column].imshow(zoom)
    for row in (0, 1):
        axes[row][column].axis("off")
figure.tight_layout()
comparison_path = save_dir / "model_comparison.png"
figure.savefig(comparison_path, dpi=150)
plt.show()
print(f"Saved {comparison_path}")

In [ ]:
# 3.3 Resource monitor: results
#
# Summarizes and plots what the cell 1.4 monitor recorded. Run it after the job finishes (Colab
# runs one cell at a time, so it cannot run while the job is still going). The log keeps being
# written while the job runs, and is also readable with `!tail /content/monitor.csv` from a Colab
# terminal if you have one.

# - The table compares the first, middle and last third of the run, so a steady decline shows up
#   as a trend.
# - Set `stop_monitor = True` to stop logging after showing the results.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

stop_monitor = False

if "log_file" not in globals():
    raise NameError("No monitor log found. Did you run cell 1.4?")

df = pd.read_csv(log_file)
for column in df.columns:
    if column != "throttle_reasons":
        df[column] = pd.to_numeric(df[column], errors="coerce")

if len(df) < 3:
    raise RuntimeError(f"Only {len(df)} samples so far; let the job run a little longer.")

df["minutes"] = df["elapsed_s"] / 60
print(f"{len(df)} samples over {df['minutes'].iloc[-1]:.1f} minutes")

# Keep only the samples taken while video2x was running, so idle time before or
# after the job does not distort the comparison
active = df[df["video2x_rss_mb"] > 0]
if len(active) >= 3:
    print(f"{len(active)} of {len(df)} samples were taken while video2x was running; using those.")
    df = active.reset_index(drop=True)
else:
    print("video2x was not seen running; using all samples.")

# Compare thirds of the run
columns = ["gpu_util_pct", "sm_clock_mhz", "temp_c", "power_w",
           "ram_used_mb", "video2x_rss_mb", "load1"]
edges = np.linspace(0, len(df), 4, dtype=int)
thirds = [df.iloc[edges[i]:edges[i + 1]] for i in range(3)]
summary = pd.DataFrame(
    {name: part[columns].mean().round(1) for name, part in
     zip(["first third", "middle third", "last third"], thirds)}
).T
display(summary)

for column, label in (("sm_clock_mhz", "SM clock"), ("ram_used_mb", "System RAM used"),
                      ("video2x_rss_mb", "video2x memory"), ("gpu_util_pct", "GPU utilization")):
    first, last = summary.loc["first third", column], summary.loc["last third", column]
    if pd.notna(first) and pd.notna(last) and first:
        print(f"{label}: {first:g} -> {last:g} ({(last - first) / first * 100:+.0f}%)")

# Decode GPU throttle reasons
THROTTLE_BITS = {
    0x1: "GPU idle", 0x2: "application clocks setting", 0x4: "software power cap",
    0x8: "hardware slowdown", 0x10: "sync boost", 0x20: "software thermal slowdown",
    0x40: "hardware thermal slowdown", 0x80: "hardware power brake",
    0x100: "display clock setting",
}
counts = {}
for value in df["throttle_reasons"].dropna():
    try:
        mask = int(str(value), 16)
    except ValueError:
        continue
    for bit, name in THROTTLE_BITS.items():
        if mask & bit:
            counts[name] = counts.get(name, 0) + 1
if counts:
    print("Throttle reasons (samples):", ", ".join(f"{k}: {v}" for k, v in sorted(counts.items())))
else:
    print("No throttle reasons reported.")

fig, axes = plt.subplots(3, 1, figsize=(10, 9), sharex=True)
axes[0].plot(df["minutes"], df["gpu_util_pct"], label="GPU utilization (%)")
axes[0].plot(df["minutes"], df["sm_clock_mhz"] / df["sm_clock_max_mhz"].max() * 100,
             label="SM clock (% of max)")
axes[0].set_ylabel("%")
axes[0].legend()
axes[1].plot(df["minutes"], df["temp_c"], label="Temperature (C)")
axes[1].plot(df["minutes"], df["power_w"], label="Power (W)")
axes[1].legend()
axes[2].plot(df["minutes"], df["ram_used_mb"], label="System RAM used (MB)")
axes[2].plot(df["minutes"], df["video2x_rss_mb"], label="video2x memory (MB)")
axes[2].set_xlabel("Minutes")
axes[2].legend()
plt.tight_layout()
plt.show()

if stop_monitor and globals().get("monitor_stop") is not None:
    monitor_stop.set()
    print("Monitor stopped.")

## 4. Experimental

Not needed for normal use. 4.1 is broken on Colab upstream. 4.2 to 4.4 test a PyTorch/CUDA version of the same network at x4, which is outside the current 1080p goal.

In [ ]:
# 4.1 Frame interpolation (RIFE, broken on Colab)
#
# Set the values below and run this cell to interpolate frames.

# - RIFE is currently broken on Colab for unknown reasons. SPIRV will fail to compile.
# - Remember free sessions can run for at most 12 hours
#   (https://research.google.com/colaboratory/faq.html#idle-timeouts). If you exceed this limit
#   your progress could be lost. You can get Colab Pro/Pro+
#   (https://colab.research.google.com/signup/pricing) for longer runtimes.
# - It may be a good idea to upscale a small sample of your video to see how well you like the
#   results before processing the whole video.

import pathlib
import subprocess

if 'input_file' not in globals() or input_file is None:
    raise NameError("Input file not specified. Did you forget to run cell 1.3?")

if 'output_dir' not in globals():
    raise NameError("Output directory not specified. Did you forget to run cell 1.3?")

# --- Interpolator Select ---
# - Choose `rife` for real-life videos.

interpolator = "rife"  # one of: rife

# Width and height of the output video. You can fill in one of them and the other one will be calculated automatically if it is left as `0`.

# --- Rife Options ---

model = "rife-v4.6"  # one of: rife, rife-HD, rife-UHD, rife-anime, rife-v2, rife-v2.3, rife-v2.4, rife-v3.0, rife-v3.1, rife-v4, rife-v4.6
frm_rate_mul = 2  # 2 to 10

# --- Advanced Options ---

# You typically don't need to touch these if you don't know what they do.

codec = "libx264"
preset = "slow"
pixfmt = "auto"
bitrate = 0
crf = 20
loglevel = "info"  # one of: trace, debug, info, warn, error, critical, none

# - `queue_size`: frames buffered between the decode, GPU and encode stages (default 4). Set it to
#   `0` for the original, non-pipelined behavior. Each buffered 4K frame uses about 12-25 MB of
#   RAM.
queue_size = 4

output_file = output_dir / f"{input_file.stem}.processed{input_file.suffix}"
if output_file.exists():
    raise FileExistsError(f"Cannot continue: file {output_file} already exists")
print(f"Output will be written to: {output_file}")

command = [
    "video2x",

    # Common options
    "--input",
    f"\"{input_file}\"",
    "--output",
    f"\"{output_file}\"",
    "--processor",
    interpolator,

    # Encoder options
    "--codec",
    codec,
    "--bit-rate",
    str(bitrate),
    "--log-level",
    loglevel,
    "--queue-size",
    str(queue_size),
    "-e",
    f"preset={preset}",
    "-e",
    f"crf={crf}"
]

# RIFE options
if interpolator == "rife":
    command.extend([
        "--rife-model",
        model,
        "--frame-rate-mul",
        str(frm_rate_mul)
    ])

if pixfmt != "auto":
    command.extend([
        "--pix-fmt",
        pixfmt
    ])

!{' '.join(command)}

In [ ]:
# 4.2 Backend test: prepare
#
# Experiment, separate from Video2X's own backends. Cells 4.2 and 4.3 measure how fast the same
# Real-ESRGAN `realesr-animevideov3` network runs through PyTorch/CUDA (and optionally ONNX
# Runtime) compared with Video2X's ncnn-Vulkan path.

# - This cell downloads the official weights, defines the network and loads it. Run it once.
# - The official weights are the x4 model. Most of the network's work happens at the input
#   resolution, so its speed is comparable to x2, but not identical.
# - `install_onnxruntime` also installs `onnx` and `onnxruntime-gpu` (used by the optional ONNX
#   Runtime test in cell 4.3).

import pathlib
import subprocess
import sys
import urllib.request

import torch
import torch.nn as nn
import torch.nn.functional as F

install_onnxruntime = True

if not torch.cuda.is_available():
    raise RuntimeError("No CUDA GPU available. Select a GPU runtime in cell 1.1.")
print(f"GPU: {torch.cuda.get_device_name(0)}   torch {torch.__version__}   CUDA {torch.version.cuda}")

if install_onnxruntime:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "onnx", "onnxruntime-gpu"], check=False
    )

weights_url = "https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.5.0/realesr-animevideov3.pth"
weights_path = pathlib.Path("/content/realesr-animevideov3.pth")
if not weights_path.exists():
    urllib.request.urlretrieve(weights_url, weights_path)

class SRVGGNetCompact(nn.Module):
    def __init__(self, num_in_ch=3, num_out_ch=3, num_feat=64, num_conv=16, upscale=4):
        super().__init__()
        self.upscale = upscale
        self.body = nn.ModuleList()
        self.body.append(nn.Conv2d(num_in_ch, num_feat, 3, 1, 1))
        self.body.append(nn.PReLU(num_parameters=num_feat))
        for _ in range(num_conv):
            self.body.append(nn.Conv2d(num_feat, num_feat, 3, 1, 1))
            self.body.append(nn.PReLU(num_parameters=num_feat))
        self.body.append(nn.Conv2d(num_feat, num_out_ch * upscale * upscale, 3, 1, 1))
        self.upsampler = nn.PixelShuffle(upscale)

    def forward(self, x):
        out = x
        for layer in self.body:
            out = layer(out)
        out = self.upsampler(out)
        return out + F.interpolate(x, scale_factor=self.upscale, mode="nearest")

checkpoint = torch.load(weights_path, map_location="cpu")
state_dict = checkpoint["params"] if "params" in checkpoint else checkpoint
feasibility_model = SRVGGNetCompact()
feasibility_model.load_state_dict(state_dict, strict=True)
feasibility_model.eval()

parameter_count = sum(p.numel() for p in feasibility_model.parameters())
print(f"Loaded realesr-animevideov3 (x4): {parameter_count / 1e6:.2f} M parameters")

In [ ]:
# 4.3 Backend test: raw network speed
#
# Compares Video2X's ncnn-Vulkan path with PyTorch/CUDA (and optionally ONNX Runtime) running the
# same network on your GPU. Run cells 1.3 and 4.2 first.

# - Baseline: `video2x --benchmark` on a short clip cut from your input (scale 4, matching the x4
#   weights). It skips encoding, but still includes decoding, colour conversion and tiling. The
#   speed is read from the summary that `video2x` prints at the `info` log level.
# - Variants: raw network time on one 1080p-sized frame, with no video decode, conversion or
#   encode. This is a best case, so treat the speedup as an upper bound.
# - Decision rule: about 2x or more over the baseline is worth considering for integration; about
#   1.3x or less is not.
# - GPU clocks fall as the T4 heats up, so the SM clock and temperature are printed after each
#   measurement.

import copy
import pathlib
import re
import subprocess
import time

import numpy as np
import pandas as pd
import torch
from IPython.display import display

if 'input_file' not in globals() or input_file is None:
    raise NameError("Input file not specified. Did you forget to run cell 1.3?")
if 'feasibility_model' not in globals():
    raise NameError("Network not loaded. Did you forget to run cell 4.2?")

clip_start = 60
clip_seconds = 5
frames_to_time = 60
frame_height = 1080
frame_width = 1920
run_onnxruntime = True

work = pathlib.Path("/content/pipeline_bench")
work.mkdir(exist_ok=True)

def gpu_state():
    result = subprocess.run(
        ["nvidia-smi", "--query-gpu=clocks.sm,temperature.gpu,power.draw", "--format=csv,noheader,nounits"],
        capture_output=True, text=True,
    )
    return result.stdout.strip().replace("\n", " ")

results = []

# --- Baseline: Video2X ncnn-Vulkan, benchmark mode (no encoding) ---
clip = work / f"feasibility_clip{input_file.suffix}"
clip.unlink(missing_ok=True)
duration = float(subprocess.run(
    ["ffprobe", "-v", "error", "-show_entries", "format=duration", "-of", "default=nw=1:nk=1", str(input_file)],
    capture_output=True, text=True, check=True,
).stdout.strip())
clip_begin = clip_start
if clip_start + clip_seconds > duration:
    clip_begin = max(0.0, duration - clip_seconds)
    print(f"The input is {duration:.1f} s long, so the clip starts at {clip_begin:.1f} s instead of {clip_start} s.")

subprocess.run(
    ["ffmpeg", "-y", "-loglevel", "error", "-ss", str(clip_begin), "-t", str(clip_seconds),
     "-i", str(input_file), "-c", "copy", str(clip)],
    check=True,
)
bench_out = work / "feasibility_bench_out.mkv"
bench = subprocess.run(
    ["video2x", "--input", str(clip), "--output", str(bench_out),
     "--processor", "realesrgan", "--realesrgan-model", "realesr-animevideov3",
     "--scaling-factor", "4", "--benchmark", "--log-level", "info", "--no-progress"],
    capture_output=True, text=True,
)
bench_out.unlink(missing_ok=True)
match = re.search(r"Average processing speed:\s*([0-9.]+)\s*FPS", bench.stdout + bench.stderr)
if not match:
    print("Could not read the baseline speed from video2x. Its output was:")
    print((bench.stdout + bench.stderr)[-1500:])
    raise RuntimeError("Baseline run failed")
baseline_fps = float(match.group(1))
if baseline_fps <= 0:
    print("video2x reported 0 FPS, so it processed no frames. Its output was:")
    print((bench.stdout + bench.stderr)[-1500:])
    raise RuntimeError("Baseline processed no frames; check the clip (is the input long enough?)")
results.append(("ncnn Vulkan (video2x --benchmark, x4)", 1000 / baseline_fps, baseline_fps, None, gpu_state()))
print(f"Baseline: {baseline_fps:.2f} fps   GPU: {gpu_state()}")

# --- PyTorch variants ---
def time_torch(dtype, channels_last, frames):
    model = copy.deepcopy(feasibility_model).to("cuda").to(dtype).eval()
    x = torch.rand(1, 3, frame_height, frame_width, device="cuda", dtype=dtype)
    if channels_last:
        model = model.to(memory_format=torch.channels_last)
        x = x.contiguous(memory_format=torch.channels_last)
    torch.backends.cudnn.benchmark = True
    torch.cuda.reset_peak_memory_stats()
    with torch.inference_mode():
        for _ in range(5):
            model(x)
        torch.cuda.synchronize()
        started = time.time()
        for _ in range(frames):
            model(x)
        torch.cuda.synchronize()
        seconds = time.time() - started
    peak_mb = torch.cuda.max_memory_allocated() / 1024 / 1024
    del model, x
    torch.cuda.empty_cache()
    return seconds / frames, peak_mb

variants = [
    ("PyTorch fp32", torch.float32, False, max(10, frames_to_time // 4)),
    ("PyTorch fp16", torch.float16, False, frames_to_time),
    ("PyTorch fp16 + channels_last", torch.float16, True, frames_to_time),
]
for name, dtype, channels_last, frames in variants:
    seconds_per_frame, peak_mb = time_torch(dtype, channels_last, frames)
    results.append((name, seconds_per_frame * 1000, 1 / seconds_per_frame, peak_mb, gpu_state()))
    print(f"{name}: {1 / seconds_per_frame:.2f} fps   GPU: {gpu_state()}")

# --- ONNX Runtime (optional) ---
if run_onnxruntime:
    try:
        import onnxruntime as ort
        if hasattr(ort, "preload_dlls"):
            ort.preload_dlls()
        model = copy.deepcopy(feasibility_model).to("cuda").half().eval()
        x = torch.rand(1, 3, frame_height, frame_width, device="cuda", dtype=torch.float16)
        onnx_path = str(work / "animevideov3_fp16.onnx")
        export_args = dict(input_names=["input"], output_names=["output"], opset_version=17)
        try:
            torch.onnx.export(model, x, onnx_path, dynamo=False, **export_args)
        except TypeError:
            torch.onnx.export(model, x, onnx_path, **export_args)
        session = ort.InferenceSession(onnx_path, providers=["CUDAExecutionProvider"])
        if "CUDAExecutionProvider" not in session.get_providers():
            raise RuntimeError("ONNX Runtime fell back to CPU (CUDA provider not available)")
        out = torch.empty(1, 3, frame_height * 4, frame_width * 4, device="cuda", dtype=torch.float16)
        binding = session.io_binding()
        binding.bind_input("input", "cuda", 0, np.float16, tuple(x.shape), x.data_ptr())
        binding.bind_output("output", "cuda", 0, np.float16, tuple(out.shape), out.data_ptr())
        for _ in range(5):
            session.run_with_iobinding(binding)
        binding.synchronize_outputs()
        started = time.time()
        for _ in range(frames_to_time):
            session.run_with_iobinding(binding)
        binding.synchronize_outputs()
        seconds_per_frame = (time.time() - started) / frames_to_time
        results.append(("ONNX Runtime CUDA fp16", seconds_per_frame * 1000, 1 / seconds_per_frame, None, gpu_state()))
        print(f"ONNX Runtime CUDA fp16: {1 / seconds_per_frame:.2f} fps   GPU: {gpu_state()}")
        del model, x, out, session, binding
        torch.cuda.empty_cache()
    except Exception as error:
        print(f"ONNX Runtime test skipped: {type(error).__name__}: {str(error)[:300]}")

table = pd.DataFrame(results, columns=["backend", "ms_per_frame", "fps", "peak_vram_mb", "gpu_clock_temp_power"])
table["speedup_vs_baseline"] = (table["fps"] / baseline_fps).round(2)
table["ms_per_frame"] = table["ms_per_frame"].round(1)
table["fps"] = table["fps"].round(2)
table["peak_vram_mb"] = table["peak_vram_mb"].round(0)
display(table)

best = table.iloc[1:]["speedup_vs_baseline"].max()
print()
if best >= 2:
    print(f"Best raw-network speedup: {best:.1f}x. Worth considering integration (still an upper bound: it excludes video I/O).")
elif best <= 1.3:
    print(f"Best raw-network speedup: {best:.1f}x. Not worth a new backend.")
else:
    print(f"Best raw-network speedup: {best:.1f}x. Marginal; an end-to-end test would be needed to decide.")

In [ ]:
# 4.4 Backend test: end-to-end CUDA pipeline
#
# Runs the same short clip through (1) Video2X's normal ncnn-Vulkan path and (2) a PyTorch/CUDA
# pipeline, both with real encoding, and compares wall-clock time and output quality. Run cells
# 1.3, 4.2 and 4.3 first.

# - Both use scale 4 (the official weights are x4) and the same `codec`, `preset` and `crf`. Audio
#   is not copied in this test, since it does not affect speed.
# - The CUDA pipeline decodes with ffmpeg, runs the network on the GPU in fp16 with
#   `channels_last`, and pipes raw RGB frames back to an ffmpeg encoder, using three threads
#   (decode, GPU, encode).
# - The CPU does the RGB to YUV conversion inside ffmpeg here, as Video2X's own path does with
#   swscale, so that cost is in both runs.
# - The Video2X run includes about a few seconds of start-up and model loading; the CUDA run's
#   model warm-up is excluded. Keep that in mind on short clips.
# - `mode`: with encoding measures the real end-to-end job. no encoding skips encoding on both
#   sides (Video2X runs with `--benchmark`, the CUDA pipeline writes to a null muxer) to compare
#   the GPU paths themselves. At 8K output the CPU encoder can be the shared bottleneck, which
#   hides any difference, so run both.
# - `compare_quality` computes PSNR between the two outputs as a sanity check. Different weights,
#   tiling or colour handling would show up as a low value.

import copy
import json
import pathlib
import queue
import re
import subprocess
import threading
import time

import pandas as pd
import torch
from IPython.display import display

if 'input_file' not in globals() or input_file is None:
    raise NameError("Input file not specified. Did you forget to run cell 1.3?")
if 'feasibility_model' not in globals():
    raise NameError("Network not loaded. Did you forget to run cell 4.2?")

clip_start = 0
clip_seconds = 5
codec = "libx264"
preset = "veryfast"
crf = 20
queue_size = 4
mode = "with encoding"  # one of: with encoding, no encoding
compare_quality = True

work = pathlib.Path("/content/pipeline_bench")
work.mkdir(exist_ok=True)

# --- Cut the clip (clamped to the input's duration) ---
duration = float(subprocess.run(
    ["ffprobe", "-v", "error", "-show_entries", "format=duration", "-of", "default=nw=1:nk=1", str(input_file)],
    capture_output=True, text=True, check=True,
).stdout.strip())
clip_begin = clip_start
if clip_start + clip_seconds > duration:
    clip_begin = max(0.0, duration - clip_seconds)
    print(f"The input is {duration:.1f} s long, so the clip starts at {clip_begin:.1f} s instead of {clip_start} s.")

clip = work / f"e2e_clip{input_file.suffix}"
clip.unlink(missing_ok=True)
subprocess.run(
    ["ffmpeg", "-y", "-loglevel", "error", "-ss", str(clip_begin), "-t", str(clip_seconds),
     "-i", str(input_file), "-c", "copy", "-an", str(clip)],
    check=True,
)

probe = json.loads(subprocess.run(
    ["ffprobe", "-v", "error", "-select_streams", "v:0", "-count_frames",
     "-show_entries", "stream=width,height,r_frame_rate,nb_read_frames", "-of", "json", str(clip)],
    capture_output=True, text=True, check=True,
).stdout)["streams"][0]
width, height = int(probe["width"]), int(probe["height"])
frame_rate = probe["r_frame_rate"]
expected_frames = int(probe["nb_read_frames"])
if expected_frames == 0:
    raise RuntimeError("The clip has no frames; check clip_start and clip_seconds")
print(f"Clip: {width}x{height}, {frame_rate} fps, {expected_frames} frames -> {width * 4}x{height * 4} output")

with_encoding = mode == "with encoding"
encoder_args = ["-c:v", codec, "-preset", preset, "-crf", str(crf), "-pix_fmt", "yuv420p"]

# --- 1. Video2X ncnn-Vulkan path (real run, with encoding) ---
ncnn_out = work / "e2e_ncnn.mp4"
ncnn_out.unlink(missing_ok=True)
print("Running Video2X (ncnn Vulkan)...")
started = time.time()
ncnn_command = [
    "video2x", "--input", str(clip), "--output", str(ncnn_out),
    "--processor", "realesrgan", "--realesrgan-model", "realesr-animevideov3",
    "--scaling-factor", "4", "--codec", codec,
    "-e", f"preset={preset}", "-e", f"crf={crf}",
    "--log-level", "warn", "--no-progress", "--queue-size", str(queue_size),
]
if not with_encoding:
    ncnn_command.append("--benchmark")
ncnn = subprocess.run(ncnn_command, capture_output=True, text=True)
ncnn_seconds = time.time() - started
if not with_encoding:
    ncnn_out.unlink(missing_ok=True)
if ncnn.returncode != 0:
    print((ncnn.stdout + ncnn.stderr)[-1500:])
    raise RuntimeError("The Video2X run failed")
print(f"  done in {ncnn_seconds:.1f} s")

# --- 2. PyTorch/CUDA pipeline ---
cuda_out = work / "e2e_cuda.mp4"
cuda_out.unlink(missing_ok=True)
out_width, out_height = width * 4, height * 4
frame_bytes = width * height * 3

model = copy.deepcopy(feasibility_model).to("cuda").half().to(memory_format=torch.channels_last).eval()
torch.backends.cudnn.benchmark = True
with torch.inference_mode():
    warmup = torch.zeros(1, 3, height, width, device="cuda", dtype=torch.float16)
    warmup = warmup.contiguous(memory_format=torch.channels_last)
    model(warmup)
    model(warmup)
    torch.cuda.synchronize()
del warmup

decoder = subprocess.Popen(
    ["ffmpeg", "-loglevel", "error", "-i", str(clip), "-f", "rawvideo", "-pix_fmt", "rgb24", "-"],
    stdout=subprocess.PIPE,
)
encoder_command = ["ffmpeg", "-y", "-loglevel", "error", "-f", "rawvideo", "-pix_fmt", "rgb24",
                   "-s", f"{out_width}x{out_height}", "-r", frame_rate, "-i", "-"]
if with_encoding:
    encoder_command += encoder_args + [str(cuda_out)]
else:
    encoder_command += ["-f", "null", "-"]
encoder = subprocess.Popen(encoder_command, stdin=subprocess.PIPE)

read_q = queue.Queue(maxsize=queue_size)
write_q = queue.Queue(maxsize=queue_size)
free_q = queue.Queue()
for _ in range(queue_size + 3):
    free_q.put(torch.empty((out_height, out_width, 3), dtype=torch.uint8).pin_memory())
errors = []

def reader():
    try:
        while True:
            data = decoder.stdout.read(frame_bytes)
            if len(data) < frame_bytes:
                break
            read_q.put(bytearray(data))
    except Exception as error:
        errors.append(error)
    finally:
        read_q.put(None)

def writer():
    broken = False
    while True:
        item = write_q.get()
        if item is None:
            break
        if not broken:
            try:
                encoder.stdin.write(memoryview(item.numpy()))
            except Exception as error:
                errors.append(error)
                broken = True
        free_q.put(item)

reader_thread = threading.Thread(target=reader, daemon=True)
writer_thread = threading.Thread(target=writer, daemon=True)

print("Running PyTorch/CUDA pipeline...")
frames = 0
wait_input = 0.0
wait_output = 0.0
wait_encoder = 0.0
started = time.time()
reader_thread.start()
writer_thread.start()
try:
    with torch.inference_mode():
        while True:
            mark = time.time()
            item = read_q.get()
            wait_input += time.time() - mark
            if item is None or errors:
                break
            x = torch.frombuffer(item, dtype=torch.uint8).view(height, width, 3).to("cuda")
            x = x.permute(2, 0, 1).unsqueeze(0).half().div_(255)
            x = x.contiguous(memory_format=torch.channels_last)
            y = model(x)
            y = y.mul_(255).round_().clamp_(0, 255).to(torch.uint8)
            y = y.permute(0, 2, 3, 1).contiguous()
            mark = time.time()
            buffer = free_q.get()
            wait_output += time.time() - mark
            buffer.copy_(y[0])
            mark = time.time()
            write_q.put(buffer)
            wait_encoder += time.time() - mark
            frames += 1
finally:
    write_q.put(None)
    writer_thread.join()
    try:
        encoder.stdin.close()
    except Exception:
        pass
    encoder.wait()
    decoder.stdout.close()
    decoder.wait()
cuda_seconds = time.time() - started

if errors or encoder.returncode != 0:
    raise RuntimeError(f"The CUDA pipeline failed: {errors or 'ffmpeg encoder exit code ' + str(encoder.returncode)}")
print(f"  done in {cuda_seconds:.1f} s ({frames} frames)")
print(f"  GPU thread time: waiting for decoded frames {wait_input / cuda_seconds * 100:.0f}%, "
      f"waiting for a free output buffer {wait_output / cuda_seconds * 100:.0f}%, "
      f"blocked handing frames to the encoder {wait_encoder / cuda_seconds * 100:.0f}%")

# --- Results ---
label = "with encoding" if with_encoding else "no encoding"
table = pd.DataFrame([
    (f"Video2X ncnn Vulkan ({label})", ncnn_seconds, expected_frames / ncnn_seconds),
    (f"PyTorch/CUDA fp16 pipeline ({label})", cuda_seconds, frames / cuda_seconds),
], columns=["backend", "seconds", "fps"])
table["speedup_vs_ncnn"] = (table["fps"] / table["fps"].iloc[0]).round(2)
table["seconds"] = table["seconds"].round(1)
table["fps"] = table["fps"].round(2)
display(table)

speedup = table["speedup_vs_ncnn"].iloc[1]
print()
if frames != expected_frames:
    print(f"Warning: the CUDA pipeline wrote {frames} frames, but the clip has {expected_frames}.")
if with_encoding:
    print(f"Speedup with encoding: {speedup:.2f}x at this output size.")
    if wait_encoder / cuda_seconds > 0.5:
        print("The GPU thread spent most of its time blocked on the encoder, so encoding and the CPU "
              "colour conversion limit both runs. Re-run with mode = no encoding to compare the GPU paths.")
else:
    if speedup >= 2:
        print(f"GPU-path speedup: {speedup:.1f}x. Worth considering, but only helps if the encoder can keep up.")
    elif speedup <= 1.3:
        print(f"GPU-path speedup: {speedup:.1f}x. Not worth a new backend.")
    else:
        print(f"GPU-path speedup: {speedup:.1f}x. Marginal.")

if with_encoding and compare_quality:
    result = subprocess.run(
        ["ffmpeg", "-i", str(ncnn_out), "-i", str(cuda_out), "-lavfi", "psnr", "-f", "null", "-"],
        capture_output=True, text=True,
    )
    match = re.search(r"average:([0-9.]+|inf)", result.stderr)
    if match:
        print(f"PSNR between the two outputs: {match.group(1)} dB (higher means more similar; "
              "below about 35 dB suggests a colour, ordering or weights mismatch).")
    else:
        print("Could not compute PSNR:", result.stderr[-500:])